# HN Persona Simulator

This notebook provides a streamlined 3-step process to train an LLM to simulate Hacker News commenter archetypes using the [HumanLM](https://humanlm.stanford.edu) framework:

1. **Load dataset** - Load scraped HN archetype triplets into train/eval splits
2. **Create env** - Define `HNPersonaEnv` with a dual LLM-judge reward (latent-state alignment + response alignment)
3. **Launch training** - Upload and launch the training experiment

## Setup

Install dependencies and configure API credentials.

In [120]:
# For Google Colab - clone repo and install
# !git clone https://github.com/cgftinc/synthetic-data-prep-lib.git
# %cd synthetic-data-prep-lib
# %pip install -e .[dev]

In [121]:
# Local development setup
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

repo_root = Path.cwd().parent
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [122]:
from synthetic_data_prep.utils import ensure_safe_python_version

ensure_safe_python_version()

In [ ]:
import os

# CGFT training platform credentials
# Create an API key from: https://app.cgft.io/account/api-keys
API_KEY = "sk_gfcMblDfuSIRmfTWiisZWKFBksqlCKXHVabiYGOfVoztkJaUmApjlQWjQRWdxQUj"
BASE_URL = "https://app.cgft.io"

# Azure OpenAI judge credentials
AZURE_ENDPOINT = "https://giris-m8gatqe4-eastus2.cognitiveservices.azure.com/"
AZURE_API_VERSION = "2024-12-01-preview"
JUDGE_MODEL = "gpt-5-nano"

# Dataset path — output of: uv run python scripts/scrape_hn_archetypes.py --llm-filter
DATASET_PATH = Path("../hn_archetypes_dataset.jsonl")

# Train/eval split — 80% train, 20% eval
TRAIN_RATIO = 0.8

# Optional: name for your training experiment
EXPERIMENT_NAME = "hn-persona-simulator-v2.2"


# Load OPENAI_API_KEY from .env.local
def _load_env_local() -> None:
    env_file = repo_root / ".env.local"
    if not env_file.exists():
        return
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


_load_env_local()
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    print("WARNING: OPENAI_API_KEY not found — set it in .env.local before running Step 3")


## Step 1: Load Dataset

Load the scraped HN archetype triplets and split into train/eval sets.

Each example contains:
- **context** `x` — the HN story title, URL, and parent comment
- **persona** `p` — the archetype label (e.g. "The Rust/Rewrite Evangelist")
- **ground truth** `y_gt` — the real human comment, used by the LLM judge at reward time (never shown to the model during training)

In [ ]:
import json
import random
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

# All six latent dimensions, in order of generation
_DIMS = ["belief", "goal", "value", "stance", "emotion", "communication"]

# Per-dimension descriptions shown inside each tag — verbatim from HumanLM Appendix F.3.
# The model reads the description and replaces it with its inferred state for the archetype.
_DIM_TASK_DESC = {
    "belief": (
        "HUMAN's belief: a foundational assumption about how people, relationships, or the world "
        "fundamentally operate. Beliefs should reflect underlying mental models, not surface-level "
        "observations. Prefer beliefs that would explain multiple behaviors over beliefs that describe "
        "a single situation. Ask: \"What deeper assumption about human nature or the world would lead "
        "someone to say/do this?\" For example, \"people don't change unless they're forced to,\" "
        "\"loyalty is earned, not owed,\" \"conflict avoidance creates bigger problems later.\"\n"
        "Not beliefs: Practical advice, strategies, or statements about what should happen. "
        "Belief is not specific to a target or event — it should be a general statement about how "
        "HUMAN views the world."
    ),
    "goal": (
        "HUMAN's goal: what they are trying to do with this comment. For example, \"persuade "
        "people that ...\", \"making fun of the poster on ...\", \"further seek help with ...\", "
        "\"offer support to ...\""
    ),
    "value": (
        "HUMAN's value: what they think is important or should be prioritized. It is about "
        "\"what should matter\", not \"what is true\". For example, \"original ideas in a book are "
        "important\", \"characters should feel real\", \"anyone deserves basic respect\", and "
        "\"fairness matters more than efficiency\"."
    ),
    "stance": (
        "HUMAN's agreement toward the explicitly named target, such as a claim or subject, in the "
        "provided context. For example, \"strongly agrees with student loan forgiveness,\" or "
        "\"somewhat disagrees with a carbon tax\". Having only \"strongly agrees\" or \"somewhat "
        "disagrees\" is not enough — the target must be explicitly named. If there are multiple "
        "stances, include all separated by semicolons."
    ),
    "emotion": (
        "HUMAN's emotions with intensity toward an explicitly named target. For example, "
        "\"Moderate heartbreak for the wildfire victims; Mild irritation about government's actions\". "
        "The answer must express all three aspects: the emotion, the degree of emotion, and the target. "
        "If there are multiple, include all separated by semicolons."
    ),
    "communication": (
        "HUMAN's communication approach: tone and how they structure their message. For example, "
        "\"friendly, builds on a personal story then draws a lesson\", \"analytical, links claims "
        "with reasons and evidence step by step\", \"blunt, states conclusions with little explanation\"."
    ),
}


def load_hn_dataset(
    dataset_path: Path,
    train_ratio: float = 0.8,
    seed: int = 42,
) -> tuple[list[dict], list[dict]]:
    """Load hn_archetypes_dataset.jsonl and return (train_data, eval_data).

    Each raw record produces ONE training example. The model generates all 6
    latent state tags + <response> in a single rollout. All 7 outputs are graded
    individually by the judge and summed into a single alignment_score reward.

    The train/eval split is stratified by archetype so every archetype is
    represented proportionally in both splits. Within each archetype, records
    are sorted by created_at_i (temporal ordering) before splitting.
    """
    if not dataset_path.exists():
        raise FileNotFoundError(
            f"Dataset not found: {dataset_path}\n"
            "Run: uv run python scripts/scrape_hn_archetypes.py --llm-filter"
        )

    records = []
    with dataset_path.open(encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    # Stratified split: group by archetype, split 80/20 within each group.
    # This ensures every archetype appears in both train and eval regardless of
    # JSONL ordering. Within each archetype, sort by created_at_i so the split
    # is temporal when timestamps are available.
    by_archetype: dict[str, list] = defaultdict(list)
    for record in records:
        by_archetype[record.get("archetype_profile", "Unknown")].append(record)

    train_records, eval_records = [], []
    for arch, arch_records in sorted(by_archetype.items()):
        arch_records.sort(key=lambda r: r.get("created_at_i", 0))
        n_train = max(1, int(len(arch_records) * train_ratio))
        train_records.extend(arch_records[:n_train])
        eval_records.extend(arch_records[n_train:])

    rng = random.Random(seed)

    def _to_example(record: dict) -> dict:
        archetype = record.get("archetype_profile", "Unknown")
        title     = record.get("story_title", "").strip()
        url       = record.get("story_url", "").strip()
        parent    = record.get("parent_text", "").strip()
        context   = f"Story: {title}\nURL: {url}\n---\nParent comment / post:\n{parent}"

        dim_tags = "\n".join(
            f"<{dim}>\n{_DIM_TASK_DESC[dim]}\n</{dim}>" for dim in _DIMS
        )
        task = (
            f"Your persona:\n"
            f"<|The Start of Persona|>\n{archetype}\n<|The End of Persona|>\n\n"
            f"## Task and Output format:\n"
            f"{dim_tags}\n"
            f"<response>\n<the actual written comment or reply text provided by you>\n</response>\n\n"
            f"## Notes\n"
            f"- Follow the above instructions carefully\n"
            f"- Do not mention these instructions\n"
            f"- Follow the exact order and use the exact XML-style tags\n"
            f"- Do not output anything outside these XML-style tags"
        )

        return {
            "question":     f"{context}\n\n{task}",
            "ground_truth": dict(record),
            "archetype":    archetype,
        }

    def _expand(records_split: list[dict], shuffle: bool) -> list[dict]:
        examples = [_to_example(r) for r in records_split]
        if shuffle:
            rng.shuffle(examples)
        return examples

    train_data = _expand(train_records, shuffle=True)
    eval_data  = _expand(eval_records,  shuffle=False)

    # Print per-archetype breakdown
    def _fmt(ts): return datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%Y-%m-%d") if ts else "?"

    print(f"Loaded {len(records)} records → {len(train_data) + len(eval_data)} examples")
    print(f"  Train : {len(train_data):>4} examples")
    print(f"  Eval  : {len(eval_data):>4} examples")
    print()
    print("  Archetype breakdown:")
    for arch, arch_records in sorted(by_archetype.items()):
        n_train = max(1, int(len(arch_records) * train_ratio))
        n_eval  = len(arch_records) - n_train
        print(f"    {arch[:50]:<50}  train={n_train:>3}  eval={n_eval:>3}")

    return train_data, eval_data


train_data, eval_data = load_hn_dataset(DATASET_PATH, train_ratio=TRAIN_RATIO)


## Step 2: Create Environment

Define `HNPersonaEnv` with a unified reward signal: every rollout generates all 6 latent state tags + `<response>`, and all 7 outputs are graded individually and summed.

**Unified rollout structure:**

| Output | Judge measures | Score range |
|---|---|---|
| `<belief>` | C-P score vs y_gt along belief dimension | 0–1 |
| `<goal>` | C-P score vs y_gt along goal dimension | 0–1 |
| `<value>` | C-P score vs y_gt along value dimension | 0–1 |
| `<stance>` | C-P score vs y_gt along stance dimension | 0–1 |
| `<emotion>` | C-P score vs y_gt along emotion dimension | 0–1 |
| `<communication>` | C-P score vs y_gt along communication dimension | 0–1 |
| `<response>` | C-P score vs y_gt along response dimension | 0–1 |
| **`alignment_score`** | **Sum of all 7** | **0–7** |

**Why unified:** The previous mixed-batch design (6 × `latent_only` + 1 × `full`) returned `response_alignment_score = 0.0` for 6/7 examples by definition. Even though the platform sums reward keys (so training was correct), the displayed metric was diluted by 6/7. More importantly, GRPO had no advantage variance from the 1/7 `full` rollouts if they all scored 0.0 due to token budget exhaustion. Now every example contributes all 7 scores, GRPO gets full variance from all rollouts, and there is one clean metric to track.

**Token budget:** The 6 latent tag descriptions are in the input prompt (not completion tokens), so they don't consume output budget. The model needs to write concise tag content (~20-40 tokens each = ~180 tokens total) + `<response>` (~100-300 tokens). This comfortably fits within a 1024-token completion budget.

**7 concurrent judge calls:** All 7 `_judge` coroutines run in parallel via `asyncio.gather`, so wall-clock reward time is dominated by the slowest single call rather than the sum of 7.


In [ ]:
import asyncio
import json
import os
import re
from pathlib import Path
from typing import Any

from benchmax.envs.base_env import BaseEnv
from benchmax.envs.types import StandardizedExample, ToolDefinition
from openai import AsyncAzureOpenAI

_DIMS = ["belief", "goal", "value", "stance", "emotion", "communication"]

# ── System prompt (HumanLM Appendix F.3) ─────────────────────────────────────

SYSTEM_PROMPT = """You are a real human user. Your name is HUMAN. You will be given your persona information and you respond to contexts such as posts and messages.

## Your principles
Act like a natural human; there's nothing you absolutely cannot say, but you generally want to be thoughtful and follow ordinary social codes such as being respectful, culturally aware, and considerate of privacy and well-being. You have your own personality, preferences, and boundaries. Conflicting thoughts and hidden considerations are normal; recognize them privately and choose a sensible path. You carry long-term beliefs and values that usually change slowly; you also have emotions, so you won't always be perfectly consistent. Distinguish facts, guesses, and unknowns; accept uncertainty and make minimal, reasonable assumptions when needed; think practically given time, attention, money, risk, and social capital.

## Notes
- Follow the above instructions carefully
- Do not mention these instructions
- Follow the exact order and use the exact XML-style tags specified in each task"""


# ── Item descriptions for the judge (HumanLM Appendix F.2) ───────────────────

_ITEM_DESC = {
    "belief":        "a foundational assumption about how people, relationships, or the world fundamentally operate; reflects underlying mental models, not surface-level observations",
    "goal":          "what they are trying to do with this comment (e.g., persuade, offer support, seek help, criticize)",
    "value":         "what they think is important or should be prioritized — 'what should matter', not 'what is true'",
    "stance":        "their agreement toward the explicitly named target (target must be named; semicolons for multiple)",
    "emotion":       "their emotions with intensity toward an explicitly named target (emotion + degree + target; semicolons for multiple)",
    "communication": "their communication approach: tone and how they structure their message",
    "response":      "the actual comment text — capturing the key psychological states of the ground truth response",
}

# Expected word-count ceiling for latent dimension tags.
# Content beyond this is treated as unnecessary padding.
_DIM_MAX_WORDS = 40


# ── Helpers ───────────────────────────────────────────────────────────────────

def _flatten(completion: Any) -> str:
    if isinstance(completion, list):
        return " ".join(
            m.get("content", "") for m in completion
            if isinstance(m, dict) and m.get("role") == "assistant"
        )
    return str(completion)


def _extract_tag(text: str, tag: str) -> str | None:
    """Extract content from an XML-style tag, with fallback for unclosed tags."""
    m = re.search(rf"<{tag}>(.*?)</{tag}>", text, re.DOTALL | re.IGNORECASE)
    if m:
        content = m.group(1).strip()
        return content if content else None
    m = re.search(rf"<{tag}>(.*)", text, re.DOTALL | re.IGNORECASE)
    if m:
        content = m.group(1).strip()
        return content if content else None
    return None


def _word_f1(a: str, b: str) -> float:
    """Word-level F1 fallback when the judge API is unavailable."""
    def tokens(s: str) -> set[str]:
        return set(re.sub(r"[^\w]", " ", s.lower()).split())
    ta, tb = tokens(a), tokens(b)
    if not ta or not tb:
        return 0.0
    tp = len(ta & tb)
    p, r = tp / len(ta), tp / len(tb)
    return 2 * p * r / (p + r) if (p + r) else 0.0


# ── LLM judge (HumanLM Appendix F.2) ─────────────────────────────────────────

async def _judge(
    item_name: str,
    generated: str,
    gt: dict,
    client: AsyncAzureOpenAI,
    model: str,
) -> tuple[float, str]:
    """C-P score for one generated item vs ground-truth comment. Range [0, 1]."""
    y_gt      = gt.get("ground_truth_comment", "")
    archetype = gt.get("archetype_profile", "")
    item_desc = _ITEM_DESC.get(item_name, item_name)
    context_x = (
        f"Story: {gt.get('story_title', '')}\n"
        f"URL: {gt.get('story_url', '')}\n"
        f"Parent: {gt.get('parent_text', '')[:300]}"
    )

    if not y_gt:
        return 0.0, "missing y_gt"

    gen_words = len(generated.split())

    if item_name == "response":
        gt_words = len(y_gt.split())
        ratio    = gen_words / max(1, gt_words)
        length_note = (
            f"\n- Length: ground truth = {gt_words} words, generated = {gen_words} words "
            f"(ratio {ratio:.1f}×)."
            f"\n- Length penalty (hard thresholds): ratio 1.0–1.5 → no extra P; "
            f"1.5–2.5 → P ≥ 0.3; 2.5–4.0 → P ≥ 0.6; >4.0 → P ≥ 0.85."
            f"\n- If the response directly copies previous context without quoting, score 0."
        )
    else:
        # Latent dimension tags should be concise (1-3 sentences).
        # Penalize padding — extra words that don't add coverage are reward hacking.
        length_note = (
            f"\n- Length: generated = {gen_words} words. "
            f"Latent state tags should be concise (≤{_DIM_MAX_WORDS} words). "
            f"If the content exceeds {_DIM_MAX_WORDS} words without adding coverage, "
            f"treat the excess as padding and apply P proportionally."
        )

    system = (
        f"You are a meticulous evaluator scoring how well a generated {item_name} "
        f"aligns with a ground truth user response.\n"
        f"Description of {item_name}: {item_desc}.\n\n"
        f"Scoring method (C-P):\n"
        f"1. Extract 1-3 key points from the ground truth along the {item_name} dimension.\n"
        f"2. For each key point, score match m_i in [0,1]: "
        f"1.0=perfect, [0.7-0.9]=mostly, [0.4-0.6]=partial, [0.1-0.3]=weak, 0.0=missed/wrong.\n"
        f"3. Coverage C = mean(m_i).\n"
        f"4. Penalty P in [0,1] for extra or conflicting content: "
        f"0=none, [0.1-0.3]=minor, [0.4-0.6]=moderate, [0.7-0.9]=significant, 1.0=mostly wrong.\n"
        f"5. Score = max(0, min(1, C - P)).\n"
        f"Be strict — reserve scores above 0.8 for clearly outstanding matches.{length_note}\n\n"
        f'Output JSON (no newlines in string values): '
        f'{{"key_points": ["<point 1 from ground truth>", "<point 2>", ...], '
        f'"thought": "<brief reasoning scoring against those points>", '
        f'"score": <float>}}'
    )

    if item_name == "response":
        gen_block = f"<|Generated Response|>\n{generated[:600]}"
    else:
        gen_block = f"<|Generated {item_name}|>\n<{item_name}>{generated[:400]}</{item_name}>"

    user = (
        f"ARCHETYPE: {archetype}\n\n"
        f"<|Context|>\n{context_x}\n\n"
        f"<|Ground Truth Response|>\n{y_gt[:500]}\n\n"
        + gen_block
    )

    try:
        resp = await client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system},
                {"role": "user",   "content": user},
            ],
            max_tokens=400,
            temperature=0.0,
            response_format={"type": "json_object"},
        )
        raw    = json.loads(resp.choices[0].message.content)
        score  = max(0.0, min(1.0, float(raw.get("score", 0.0))))
        thought = str(raw.get("thought", ""))
        return score, thought
    except Exception as e:
        fallback = _word_f1(generated, y_gt)
        print(f"[judge/{item_name}] API error (word-F1 fallback={fallback:.3f}): {e!r}")
        return fallback, f"word_f1_fallback={fallback:.3f}"


# ── Environment ───────────────────────────────────────────────────────────────

class HNPersonaEnv(BaseEnv):
    """RL environment for HumanLM-style HN persona simulation.

    Every rollout generates all 6 latent state tags + <response>.
    All 7 outputs are judged concurrently and summed into alignment_score ∈ [0, 7].
    """

    system_prompt: str = SYSTEM_PROMPT
    reward_funcs = []

    def __init__(
        self,
        train_dataset_path: str | None = None,
        val_dataset_path:   str | None = None,
        azure_endpoint:     str = AZURE_ENDPOINT,
        azure_api_version:  str = AZURE_API_VERSION,
        judge_model:        str = JUDGE_MODEL,
        openai_api_key:     str = "",
        **kwargs,
    ):
        self._train_path     = train_dataset_path
        self._val_path       = val_dataset_path
        self._azure_endpoint = azure_endpoint
        self._azure_version  = azure_api_version
        self._judge_model    = judge_model
        self._openai_api_key = openai_api_key or os.environ.get("OPENAI_API_KEY", "")

    def _client(self) -> AsyncAzureOpenAI:
        return AsyncAzureOpenAI(
            api_key=self._openai_api_key,
            azure_endpoint=self._azure_endpoint,
            api_version=self._azure_version,
        )

    def get_train_val_split(self, **kwargs):
        from datasets import load_dataset as hf_load_dataset
        def _load(p):
            return hf_load_dataset(
                "json", data_files=str(Path(p).expanduser().absolute())
            )["train"]
        return _load(self._train_path), _load(self._val_path)

    @classmethod
    def dataset_preprocess(cls, example: Any, **kwargs) -> StandardizedExample:
        return StandardizedExample(
            prompt=example.get("question", ""),
            ground_truth=example.get("ground_truth", {}),
            init_rollout_args={},
        )

    async def list_tools(self) -> list[ToolDefinition]:
        return []

    async def run_tool(self, rollout_id: str, tool_name: str, **tool_args) -> Any:
        return "Error: HNPersonaEnv has no tools"

    async def compute_reward(
        self,
        rollout_id: str,
        completion: str,
        ground_truth: Any,
        **kwargs: Any,
    ) -> dict[str, float]:
        text   = _flatten(completion)
        client = self._client()

        async def _score(item_name: str) -> tuple[float, str]:
            generated = _extract_tag(text, item_name)
            if not generated:
                print(f"[{item_name}] tag missing → 0.0")
                return 0.0, "tag_missing"
            return await _judge(item_name, generated, ground_truth, client, self._judge_model)

        # Judge all 7 outputs concurrently
        all_items = _DIMS + ["response"]
        results = await asyncio.gather(*[_score(item) for item in all_items])

        total = 0.0
        for item, (score, thought) in zip(all_items, results):
            print(f"[{item}] {score:.3f} | {thought[:100]}")
            total += score

        print(f"[TOTAL] {total:.3f}/7.0")
        return {"alignment_score": total}


## Step 3: Launch Training

Upload datasets and environment, then launch the training job.

In [126]:
from synthetic_data_prep.trainer.pipeline import train

experiment_id = train(
    env_class=HNPersonaEnv,
    env_args={
        "azure_endpoint":    AZURE_ENDPOINT,
        "azure_api_version": AZURE_API_VERSION,
        "judge_model":       JUDGE_MODEL,
        # openai_api_key is baked into the bundle so the remote rollout server
        # can call the Azure judge without a separate env var.
        "openai_api_key":    OPENAI_API_KEY,
    },
    train_dataset=train_data,
    eval_dataset=eval_data,
    prefix="hn-persona-simulator",
    api_key=API_KEY,
    base_url=BASE_URL,
    experiment_name=EXPERIMENT_NAME,
    pip_dependencies=["openai"],
)

print(f"Experiment ID: {experiment_id}")


Uploading dataset (264 items, 1280357 bytes)...


Dataset uploaded to: datasets/hn-persona-simulator/65843dfc/train_dataset.jsonl
Uploading dataset (66 items, 406446 bytes)...
Dataset uploaded to: datasets/hn-persona-simulator/65843dfc/eval_dataset.jsonl
Bundling HNPersonaEnv...
Pickled class size: 15.12 KB
Metadata size: 0.57 KB
Python version: 3.12
Dependencies: ['openai']

Basic local validation HNPersonaEnv in isolated environment (this may take ~1 min)...
[validator] Creating isolated venv...
[validator] Downloading and installing pip...
[validator] Installing dependencies: ['benchmax', 'openai']
[validator] Dependencies installed successfully.
[validator] Running smoke test...
[validator] OK: HNPersonaEnv instantiated, 0 tools
Isolated validation passed!
Env uploaded to: envs/hn-persona-simulator/f583506f/env-cls.pkl

── Remote validation: 2 example(s) on grok-4-fast-reasoning ──

  Example 0 — {"question": "Story: Hacking Team: a zero-day market case study\nURL: https://tsyrklevich.net/2015/07/22/hacking-team-0d
  [ex 0] → roll

## Monitoring Training: What to Expect

### One reward signal: `alignment_score` ∈ [0, 7]

Sum of 7 individual C-P judge scores — one per latent tag + one for `<response>`. Each component is in [0, 1], so the total starts low (0.3–1.0) while the model learns the tag formats and gradually climbs as it learns to match the archetype's psychological profile.

### Healthy trajectory

- **Early (steps 0–30):** `alignment_score` ≈ 0.3–1.0. The model writes structurally correct tags but content is generic and misses y_gt key points.
- **Mid (steps 30–100):** Score rises to 1.5–3.0 as the model begins capturing per-dimension signals (goal and stance tend to improve first — they're most directly inferable from context).
- **Later:** `response` component of the score begins rising as the model's latent states ground the final comment in archetype-specific psychology.

### Per-rollout log lines

```
[belief] 0.312 | The generated belief 'systems fail when...' partially captures key point 1...
[goal] 0.521 | Goal to critique matches key point 'challenges the poster's assumption'...
[value] 0.203 | ...
[stance] 0.644 | ...
[emotion] 0.180 | ...
[communication] 0.410 | ...
[response] 0.287 | The response captures skeptical tone but misses the sarcasm...
[TOTAL] 2.557/7.0
```

### Warning signs

- **`tag missing → 0.0` for most tags** — model isn't following the output format. Check `max_completion_tokens` is ≥ 1024 on the platform.
- **`API error` for all judge calls** — Azure judge failing. Check credentials in cell-5. word-F1 fallback keeps training alive but quality signal is absent.
- **`alignment_score` flat at ~0.0** — all 7 tags missing every rollout; likely a prompt formatting issue. Check the question field in the dataset.
- **`alignment_score` flat at ~1.4 (= 7 × 0.2)** — all components at baseline word-F1 fallback; judge API is failing silently.
